In [ ]:
#Retirar comentário para limpar o diretório caso necessário
"""import os
os.chdir("/content")
!rm -rf /content/crise-saude-ufam"""

In [ ]:
import json, time, datetime, pathlib, unicodedata, shutil, glob, os
from getpass import getpass
from google.colab import userdata
from pathlib import Path
import pyarrow.parquet as pq
import httpx

import requests
import pandas as pd

print(f"pandas  {pd.__version__}")
print(f"requests {requests.__version__}")

#Autenticação e Clonagem do Repositório Github

In [92]:

from getpass import getpass

TOKEN = userdata.get('GITHUB_TOKEN')

%cd /content

!git clone -b Gabriel-branch https://{TOKEN}@github.com/victorbaraunaAcad/crise-saude-ufam.git
%cd crise-saude-ufam

!git config user.name "gabrielsanttos"
!git config user.email "gabrielconceicao827@gmail.com"

/content
fatal: destination path 'crise-saude-ufam' already exists and is not an empty directory.
/content/crise-saude-ufam


In [ ]:
%%script echo "Célula desativada: fazia git commit/push toda vez que o notebook rodava. Rode à mão só se precisar."

%cd /content/crise-saude-ufam

# Apaga o registro do submódulo/pasta duplicada do rastreamento do Git
!git rm --cached crise-saude-ufam -f 2>/dev/null || true

# Envia a correção para a sua branch no GitHub
!git commit -m "Remove pasta duplicada e referencia de submodulo"
!git push origin Gabriel-branch

In [ ]:
!git remote -v
!git status
!git config --list

#Configuração de Diretórios do Projeto

In [ ]:
RAIZ = pathlib.Path.cwd()
if RAIZ.name == "notebooks":          # permite rodar de dentro de notebooks/ (igual ao 00_coleta_atlas)
    RAIZ = RAIZ.parent
BRUTOS   = RAIZ / "data" / "raw"
TRATADOS = RAIZ / "data" / "processed"
BRUTOS.mkdir(parents=True, exist_ok=True); TRATADOS.mkdir(parents=True, exist_ok=True)
print("Trabalhando em:", RAIZ)

In [ ]:
#script echo "Ignorando comandos de Git"
%cd /content/crise-saude-ufam
!git remote set-url origin https://github.com/victorbaraunaAcad/crise-saude-ufam.git

#Configuração de Remoto do Git e Git LFS

In [ ]:
#script echo "Ignorando comandos de Git"
!apt-get update && apt-get install git-lfs -y

%cd /content/crise-saude-ufam
!git lfs install
!git lfs pull
!git pull

# Coleta de Dados Extras para a Base

In [ ]:

%pip install pysus
import pysus

%pip install nest_asyncio -q
import nest_asyncio
nest_asyncio.apply()

%pip install duckdb -q
import duckdb
con = duckdb.connect()

print(f"pysus {pysus.get_version()}")

In [ ]:
ARQ_PROV = RAIZ / "docs" / "proveniencia.csv"
ARQ_PROV.parent.mkdir(parents=True, exist_ok=True)
RAIZ_BRUTOS = BRUTOS          # antes era um caminho fixo do Colab (/content/...)

def registrar(fonte, url, metodo, parametros, n_linhas, arquivo_bruto):
    """Mesma função do 00_coleta_atlas, com a trava que não duplica a linha se o notebook
    for rodado de novo. Única diferença: aceita uma LISTA de arquivos (gravados separados por '; ')."""
    arquivos = arquivo_bruto if isinstance(arquivo_bruto, (list, tuple)) else [arquivo_bruto]
    rel = "; ".join(str(pathlib.Path(a).relative_to(RAIZ)) for a in arquivos)
    if ARQ_PROV.exists() and ARQ_PROV.stat().st_size > 0:
        ja = pd.read_csv(ARQ_PROV)
        if ((ja["fonte"] == fonte) & (ja["arquivo_bruto"] == rel)).any():
            print(f"  (proveniência já tinha: {fonte})")
            return
    linha = pd.DataFrame([{
        "fonte": fonte, "url": url, "metodo": metodo,
        "parametros": json.dumps(parametros, ensure_ascii=False),
        "n_linhas": n_linhas, "arquivo_bruto": rel,
        "coletado_em": datetime.datetime.now().astimezone().isoformat(timespec="seconds"),
    }])
    cabecalho = not ARQ_PROV.exists() or ARQ_PROV.stat().st_size == 0
    linha.to_csv(ARQ_PROV, mode="a", header=cabecalho, index=False)
    print(f"  ✓ proveniência: {fonte} ({n_linhas} linhas)")

def normalizar(s):
    if pd.isna(s): return ""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return s.upper().strip()

# Trava: este notebook NÃO altera a base consolidada original.
# Guardamos o hash do arquivo agora e conferimos no final do notebook.
import hashlib
ARQ_BASE_ORIGINAL = TRATADOS / "base_consolidada_municipios.csv"
def hash_arquivo(arq):
    return hashlib.sha256(arq.read_bytes()).hexdigest() if arq.exists() else None
HASH_BASE_ORIGINAL = hash_arquivo(ARQ_BASE_ORIGINAL)
print("Hash da base original guardado:", (HASH_BASE_ORIGINAL or "arquivo não encontrado")[:12])

In [ ]:
#Definindo as funções que faz o download dos dados
from pysus import list_files

CODIGO_UF = {
    "AM": "13", "RS": "43"
}

def baixar_do_estado(nome_base, funcao, uf, ano, coluna_municipio="CODMUNRES"):
    """Baixa a base e garante que só sobre o estado pedido."""
    catalogo = pysus.ftp.list_files(dataset=nome_base, state=uf, year=ano)
    do_estado = {
        pathlib.Path(str(n)).name
        for n in catalogo.loc[catalogo["state"] == uf, "name"]
    }
    #salva a url de onde os dados foram baixados
    urls = [f"ftp://ftp.datasus.gov.br/{n}"
            for n in catalogo.loc[catalogo["state"] == uf, "name"]]


    caminhos = funcao(state=uf, year=ano)
    escolhidos = [p for p in caminhos if pathlib.Path(str(p)).name in do_estado]

    if escolhidos:
        df = pd.concat([pd.read_parquet(p) for p in escolhidos], ignore_index=True)
        arquivos_para_copiar = escolhidos
    else:
        df = pd.concat([pd.read_parquet(p) for p in caminhos], ignore_index=True)
        antes = len(df)
        df = df[df[coluna_municipio].astype(str).str[:2] == CODIGO_UF[uf]]
        print(f"  (este ano só tem arquivo nacional: filtrei {antes:,} linhas "
              f"do país para {len(df):,} de {uf})")
        arquivos_para_copiar = caminhos   # antes copiava "escolhidos", que aqui é vazio


    df = df.reset_index(drop=True)

    # copia os arquivos originais do PySUS (tal como vieram) para data/raw/
    caminhos_copiados = []
    for origem in arquivos_para_copiar:
        destino = RAIZ_BRUTOS / pathlib.Path(str(origem)).name
        shutil.copy(origem, destino)
        caminhos_copiados.append(str(destino))


    registrar(
        fonte=nome_base,
        url="; ".join(urls),
        metodo=funcao.__name__,
        parametros={"state": uf, "year": ano},
        n_linhas=len(df),
        arquivo_bruto=caminhos_copiados,
    )

    return df


#Função para baixar dados de várias UFs de uma vez e concatenar, utilizada para
def baixar_regiao(nome_base, funcao, ufs, ano, coluna_municipio="CODMUNRES"):
    """Baixa e junta os dados de várias UFs em uma única tabela."""
    dfs = []
    for uf in ufs:
        print(f"Baixando {nome_base} - {uf} - {ano}...")
        df_uf = baixar_do_estado(nome_base, funcao, uf, ano, coluna_municipio)
        df_uf["UF"] = uf  # garante rastreabilidade, mesmo que o código já traga o estado
        dfs.append(df_uf)
    return pd.concat(dfs, ignore_index=True)

In [ ]:
#Definição das funções que realizam os downloads e checagem dos meses com informações
#sobre internações dos anos que faltavam: AM 2022 e RS 2022-2023
#Base utilizada: SIH(Sistema de Informações Hospitalares)
from pysus import list_files
#Primeiro verificamos os meses disponíveis para download para cada localidade e ano

def meses_com_internacoes(uf,ano):
  catalogo = pysus.ftp.list_files(dataset="sih", state=uf, year=ano)
  # O nome do arquivo é do tipo RDPR2408.parquet: grupo(2) + UF(2) + ano(2) + mês(2)
  meses_rd = sorted(
    int(nome.split( "\\" ) [-1][6:8])
    for nome in catalogo["name"]
    if nome.split("\\")[-1].startswith("RD")
)

  print(f"Meses com internações (grupo RD) em {uf}/{ano}: {meses_rd}")

  if not meses_rd:
      print("⚠ Nenhum mês publicado para este ano. Tente o ano anterior.")
  else:
      MES = meses_rd[-1]     # usamos o mês mais recente disponível



meses_com_internacoes("AM", 2022)
meses_com_internacoes("RS", 2022)
meses_com_internacoes("RS", 2023)



def baixa_meses_com_internacoes_uma_uf(uf, ano):
    """Baixa todos os meses de internação (RD) disponíveis para uma UF/ano e consolida em um único arquivo."""
    catalogo = list_files(dataset="sih", state=uf, year=ano)

    # set(): com vários anos o mesmo mês aparece uma vez por ano
    meses_rd = sorted({
        int(nome.split("\\")[-1][6:8])
        for nome in catalogo["name"]
        if nome.split("\\")[-1].startswith("RD")
    })
    if not meses_rd:
        raise ValueError(f"Nenhum mês com arquivo RD encontrado para {uf} em {ano}.")

    urls = [
        f"ftp://ftp.datasus.gov.br/{n}"
        for n in catalogo.loc[catalogo["name"].str.split("\\").str[-1].str.startswith("RD"), "name"]
    ]

    caminhos = sih(state=uf, year=ano, month=meses_rd)
    arquivos_rd = [c for c in caminhos if os.path.basename(str(c)).upper().startswith("RD")]
    if not arquivos_rd:
        raise ValueError(f"Nenhum arquivo RD baixado para {uf} em {ano}.")

    internacoes = pd.concat(
        [pd.read_parquet(str(c).replace("\\", "/")) for c in arquivos_rd],
        ignore_index=True
    )

    print(f"{len(internacoes):,} internações em {uf}, meses {meses_rd} de {ano}")
    print(f"{len(internacoes.columns)} colunas")

    for origem in arquivos_rd:
        destino = RAIZ_BRUTOS / pathlib.Path(str(origem)).name
        shutil.copy(origem, destino)

    # [2022, 2023] vira "2022_2023" (antes o nome ficava "sih_am_[2022, 2023].parquet")
    rotulo_ano = "_".join(map(str, ano)) if isinstance(ano, (list, tuple)) else str(ano)
    arquivo_bruto = RAIZ_BRUTOS / f"sih_{uf.lower()}_{rotulo_ano}.parquet"
    internacoes.to_parquet(arquivo_bruto, index=False)

    registrar(
        fonte="SIH",
        url="; ".join(urls),
        metodo="sih",
        parametros={"state": uf, "year": ano, "month": meses_rd},
        n_linhas=len(internacoes),
        arquivo_bruto=str(arquivo_bruto),
    )

    return internacoes


def baixa_meses_com_internacoes(ufs, ano):
    """Aceita uma UF (string) ou uma lista de UFs, e devolve tudo já concatenado."""
    if isinstance(ufs, str):
        ufs = [ufs]

    dfs = []
    for uf in ufs:
        print(f"Baixando internações - {uf} - {ano}...")
        df_uf = baixa_meses_com_internacoes_uma_uf(uf, ano)
        df_uf["UF"] = uf
        dfs.append(df_uf)

    return pd.concat(dfs, ignore_index=True)

In [ ]:
#Coleta dos anos de internações que faltavam na base: AM 2022 e RS 2022-2023
from pysus import sih
# ATENÇÃO: sih() devolve QUATRO arquivos — RD (as internações), SP (serviços
# profissionais), RJ e ER (rejeitadas)

#Primeira coleta: UF = Amazonas, Período: 2022  (a base original já tem 2023-2024 do AM)
internacoes_am_2022 = baixa_meses_com_internacoes_uma_uf("AM", 2022)

#Segunda coleta: UF = Rio Grande do Sul, Período: [2022,2023]  (a base original já tem 2024 do RS)
internacoes_rs_2022_2023 = baixa_meses_com_internacoes_uma_uf("RS", [2022,2023])




In [ ]:
#Segunda coleta: SINAN (Sistema de Informação de Agravos de Notificação)
#Definição das funções utilizadas


def mil(numero):
  "Número no padrão brasileiro: 1234567 vira 1.234.567."
  return f"{numero:,.0f}".replace(",", ".")


def _arquivo_legivel(caminho):
    "Confere se TODOS os blocos do parquet abrem — count(*) não basta."
    try:
        leitor = pq.ParquetFile(caminho)
        for grupo in range(leitor.metadata.num_row_groups):
            leitor.read_row_group(grupo)
        return True
    except Exception:
        return False


def _datas_convertem(caminho, coluna="DT_SIN_PRI"):
    """Confere se a coluna de data está em formato ISO e não compacto."""
    try:
      caminho_sql = str(caminho).replace("\\", "/")
      total, iso_ok, compacto_ok = duckdb.sql(f"""
                  SELECT
                  count(*),
                  count(TRY_CAST({coluna} AS DATE)),
                  count(TRY_STRPTIME({coluna}, '%Y%m%d'))
              FROM (SELECT {coluna} FROM read_parquet('{caminho_sql}') LIMIT 5000)
          """).fetchone()
    except Exception:
          return True
    if total == 0:
          return True
    return max(iso_ok, compacto_ok) >= total * 0.9


def arquivo_nacional_confiavel(sigla, ano):
    """Baixa (por fluxo único) e confere o arquivo nacional do agravo no SINAN.
    Contorna o defeito medido (31/08–01/09/2026) de corrida entre duas
    origens duplicadas do catálogo. Retorna (caminho_local, caminho_remoto)."""
    import pysus

    catalogo = pysus.list_files(dataset="sinan", year=ano)
    catalogo = catalogo.assign(arquivo=[
        os.path.basename(str(n).replace("\\", "/")) for n in catalogo["name"]])
    candidatos = catalogo[catalogo["arquivo"].str.upper()
                          .str.startswith(sigla.upper())]
    if candidatos.empty:
        raise FileNotFoundError(f"sem arquivo nacional de {sigla} para {ano}")

    preferida = candidatos[candidatos["path"].astype(str).str.contains("dadosgov")]
    escolhido = (preferida if len(preferida) else candidatos).iloc[0]

    destino = (Path(pysus.CACHEPATH) / "downloads" / "ducklake" / "sinan"
               / escolhido["arquivo"])
    destino.parent.mkdir(parents=True, exist_ok=True)

    if not (destino.exists() and _arquivo_legivel(destino)
            and _datas_convertem(destino)):
        motivo = ("a cópia local estava corrompida ou com datas compactas"
                  if destino.exists() else "primeira vez nesta máquina")
        print(f"  Baixando {escolhido['arquivo']} por fluxo único ({motivo})…")
        from pysus.api import types
        origem = (f"https://{types.S3_ENDPOINT}/{types.S3_BUCKET}/"
                  + str(escolhido["path"]).replace("\\", "/"))
        with httpx.Client(timeout=httpx.Timeout(20, read=180)) as cliente, \
             cliente.stream("GET", origem) as resposta, \
             open(destino, "wb") as saida:
            resposta.raise_for_status()
            for pedaco in resposta.iter_bytes(chunk_size=1024 * 1024):
                saida.write(pedaco)
        if not _datas_convertem(destino):
            raise RuntimeError(
                f"{escolhido['arquivo']} veio com datas compactas mesmo da "
                "origem preferida — as duas origens divergiram de novo.")
        if not _arquivo_legivel(destino):
            raise RuntimeError(
                f"{escolhido['arquivo']} veio ilegível mesmo por fluxo único — "
                "a origem está servindo o arquivo corrompido. Tente mais tarde.")

    caminho_remoto = f"https://.../{str(escolhido['path']).replace(chr(92)+chr(92), '/')}"
    return str(destino).replace("\\", "/"), str(escolhido["path"]).replace("\\", "/")


def baixar_sinan_doencas(doencas, ano,uf_filtro=None):
    """Baixa, confere e consolida o SINAN para um conjunto de doenças.

    doencas: dict {sigla: nome legível}, ex.: {"LEPT": "Leptospirose", "DENG": "Dengue"}
    Salva o bruto conferido + o consolidado padronizado em RAIZ_BRUTOS,
    e registra a proveniência de cada doença.
    """
    from pysus.api import types
    COLUNAS_UTEIS = ["SG_UF", "SG_UF_NOT", "ID_MUNICIP", "ID_MN_RESI", "DT_NOTIFIC", "DT_SIN_PRI", "EVOLUCAO"]

    resultados = {}
    for sigla, nome in doencas.items():
        print(f"Baixando SINAN - {nome} ({sigla}) - {ano}...")
        try:
            caminho_local, caminho_remoto = arquivo_nacional_confiavel(sigla, ano)
        except FileNotFoundError as e:
            print(f"  ⚠ pulando {sigla}: {e}")
            continue

        # lê já filtrado, sem carregar o nacional inteiro em pandas
        if uf_filtro:
            df = ler_filtrado_por_uf(caminho_local, uf_filtro, colunas=COLUNAS_UTEIS)
        else:
            df = pd.read_parquet(caminho_local, columns=COLUNAS_UTEIS)

        print(f"  {len(df):,} registros, {len(df.columns)} colunas")

        destino_bruto = RAIZ_BRUTOS / os.path.basename(caminho_local)
        shutil.copy(caminho_local, destino_bruto)  # bruto nacional continua intacto em disco

        arquivo_tratado = RAIZ_BRUTOS / f"sinan_{sigla.lower()}_{ano}.parquet"
        df.to_parquet(arquivo_tratado, index=False)

        registrar(
            fonte=f"SINAN - {nome}",
            url=f"https://{types.S3_ENDPOINT}/{types.S3_BUCKET}/{caminho_remoto}",
            metodo="arquivo_nacional_confiavel (fluxo único, contorna corrida do catálogo)",
            parametros={"disease": sigla, "year": ano},
            n_linhas=len(df),
            arquivo_bruto=str(arquivo_tratado),
        )

        resultados[sigla] = df

    return resultados

In [ ]:
##A função sinan() do PySUS só aceita disease e year como parâmetros
#— não tem state nenhum, diferente de sih()/sim(), que aceitam state.
#Isso não é limitação do PySUS, é reflexo de como o DATASUS
#disponibiliza o dado: o SINAN é publicado como um arquivo único por agravo,
#cobrindo o Brasil inteiro.

#O período de coleta será o mesmo [2022,2023]
DOENCAS = {"LEPT": "Leptospirose", "DENG": "Dengue", "ZIKAB": "Zika", "CHIKB": "Chikungunya"}

dados_sinan_2022 = baixar_sinan_doencas(DOENCAS, 2022)
dados_sinan_2023 = baixar_sinan_doencas(DOENCAS,2023)

df_lept = dados_sinan_2022["LEPT"]


In [ ]:
#Terceira Coleta: SIM (Sitema de Informações sobre Mortalidade)
from pysus import sim

#UF = Amazonas, Período: 2022  (a base original já tem 2023-2024 do AM)
obitos_am_2022 = baixar_do_estado("sim", sim, "AM", 2022)

#UF = Rio Grande do Sul, Período: [2022,2023]
obitos_rs_2022_2023 = baixar_do_estado("sim", sim, "RS", [2022, 2023])


#Informações sobre os arquivos baixados
print(f"{len(obitos_am_2022):,} óbitos registrados em AM, 2022")
print(f"{len(obitos_am_2022.columns)} colunas disponíveis")

print(f"{len(obitos_rs_2022_2023):,} óbitos registrados em RS, 2022-2023")
print(f"{len(obitos_rs_2022_2023.columns)} colunas disponíveis")



### Funções de Verificação e Inspeção de Dados

Definição de funções para verificar a integridade das tabelas coletadas

In [ ]:
def analisar_tabela(df, nome_tabela):
    """Exibe informações básicas, dimensões e primeiras linhas de um DataFrame."""
    print(f"=== Análise da Tabela: {nome_tabela} ===")
    print(f"Dimensões: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas\n")

    # Tipos de dados e valores não-nulos
    print("--- Resumo de Tipos e Completude (Amostra de colunas) ---")
    info_df = pd.DataFrame({
        'Tipo': df.dtypes,
        'Preenchidos (%)': (df.notna().sum() / len(df) * 100).round(2),
        'Valores Únicos': df.nunique()
    })
    display(info_df.head(15))
    print("\n--- Primeiras 5 linhas ---")
    display(df.head(5))
    print("=" * 50 + "\n")

def verificar_cobertura_temporal(df, coluna_data, formato="%Y%m%d"):
    """Analisa o intervalo temporal coberto pelos dados na tabela."""
    try:
        datas = pd.to_datetime(df[coluna_data], format=formato, errors='coerce')
        print(f"Intervalo Temporal ({coluna_data}):")
        print(f"  Mínimo: {datas.min()}")
        print(f"  Máximo: {datas.max()}")
        print(f"  Registros nulos na data: {datas.isna().sum():,}")
    except Exception as e:
        print(f"Erro ao analisar datas na coluna {coluna_data}: {e}")

In [ ]:
# Exemplo de verificação na tabela de Leptospirose
analisar_tabela(df_lept, "SINAN - Leptospirose 2022")
verificar_cobertura_temporal(df_lept, "DT_NOTIFIC")

In [ ]:
# Verificação na tabela de Internações do Amazonas
analisar_tabela(internacoes_am_2022, "SIH - Internações Amazonas 2022")
verificar_cobertura_temporal(internacoes_am_2022, "DT_INTER")

In [ ]:
# Verificação na tabela de Internações do Rio Grande do Sul
analisar_tabela(internacoes_rs_2022_2023, "SIH - Internações Rio Grande do Sul 2022-2023")
verificar_cobertura_temporal(internacoes_rs_2022_2023, "DT_INTER")

In [ ]:
import pandas as pd

# Lendo o arquivo de proveniência para verificar as coletas registradas
try:
    df_prov = pd.read_csv(ARQ_PROV)
    print(f"O arquivo possui {len(df_prov)} registros de proveniência.")
    display(df_prov)
except Exception as e:
    print(f"Erro ao abrir o arquivo {ARQ_PROV}: {e}")

# Tabela mensal DATASUS × município (2022–2024)

O Atlas (`00_coleta_atlas` → `atlas_eventos_am_rs_2022_2024`) marca **em que mês** cada município do AM e do RS teve seca ou enchente. Para cruzar com ele, o DATASUS precisa estar no mesmo formato: **uma linha por município e mês**.

### Decisões

* **Não altera a base original.** `base_consolidada_municipios.csv` não é lida nem gravada aqui. A saída é uma tabela nova, `data/processed/datasus_mensal_am_rs_2022_2024.csv` (e `.parquet`). Uma trava confere no final que o arquivo da base continua idêntico (hash SHA-256).
* **Universo:** os 62 municípios do AM e os 497 do RS (`municipios_am.csv` e `municipios_rs.csv`, as mesmas tabelas que o Atlas usa para conferir), de 2022-01 a 2024-12. Município/mês sem registro recebe 0.
* **Anos:** 2023–2024 do AM e 2024 do RS vêm da primeira coleta (já estão em `data/raw`); 2022 do AM e 2022–2023 do RS vêm desta coleta. Meses do SIH que faltarem em `data/raw` são baixados.
* **Sem contagem dupla:** o SIH é lido só dos arquivos mensais `RD{UF}{AA}{MM}.parquet`, nunca dos `sih_*.parquet` consolidados (que repetem as mesmas linhas).
* **Mês de referência = mês do acontecimento**, não o mês de processamento do arquivo: `DT_INTER` (SIH), `DTOBITO` (SIM) e `DT_SIN_PRI` (SINAN; `DT_NOTIFIC` quando vazia).
* **Chave:** `codigo_ibge_6` (`codigo_ibge // 10`), a mesma do Atlas.
* **Município de residência** (`MUNIC_RES`, `CODMUNRES`, `ID_MN_RESI`), como na base original.

In [ ]:
from pysus import sih

UFS  = ["AM", "RS"]
ANOS = [2022, 2023, 2024]
MESES = list(pd.period_range(f"{ANOS[0]}-01", f"{ANOS[-1]}-12", freq="M").strftime("%Y-%m"))
SAIDA_MENSAL = TRATADOS / "datasus_mensal_am_rs_2022_2024"

# Universo de municípios: as mesmas tabelas que o 00_coleta_atlas usa na conferência
municipios = pd.concat(
    [pd.read_csv(TRATADOS / f"municipios_{uf.lower()}.csv").assign(uf=uf) for uf in UFS],
    ignore_index=True,
).rename(columns={"municipio_id": "codigo_ibge", "municipio_nome": "nome_municipio"})
municipios["codigo_ibge"] = municipios["codigo_ibge"].astype("int64")
municipios["codigo_ibge_6"] = municipios["codigo_ibge"] // 10
municipios = municipios[["codigo_ibge", "codigo_ibge_6", "nome_municipio", "uf"]]
assert municipios["codigo_ibge_6"].is_unique
print("Municípios por UF:", municipios.groupby("uf").size().to_dict())   # esperado: {'AM': 62, 'RS': 497}
print(f"{len(MESES)} meses: {MESES[0]} a {MESES[-1]}")


def para_mes(serie):
    """Converte as datas do DATASUS para 'AAAA-MM'.
    Aceita ISO (2022-01-31), AAAAMMDD (SIH) e DDMMAAAA (SIM). Para anos 20xx os formatos
    não se confundem: o 'mês' lido no formato errado sempre sai inválido (20, 22, 23…)."""
    if pd.api.types.is_datetime64_any_dtype(serie):
        return serie.dt.strftime("%Y-%m")
    s = serie.astype("string").str.strip()
    datas = pd.to_datetime(s.str[:10], format="%Y-%m-%d", errors="coerce")
    for formato in ("%Y%m%d", "%d%m%Y"):
        datas = datas.fillna(pd.to_datetime(s, format=formato, errors="coerce"))
    return datas.dt.strftime("%Y-%m")


def codigo6(serie):
    "Município com 6 dígitos (o DATASUS usa o código do IBGE sem o dígito verificador)."
    return pd.to_numeric(serie.astype("string").str.strip().str[:6], errors="coerce")

In [ ]:
def garantir_rd(uf, ano):
    """Confere se os 12 arquivos mensais RD{UF}{AA}{MM}.parquet estão em data/raw e baixa os que
    faltarem (a primeira coleta, por exemplo, ficou sem RDAM2303 e RDAM2402).
    Devolve (arquivos existentes, meses que continuam faltando)."""
    esperados = {m: RAIZ_BRUTOS / f"RD{uf}{str(ano)[2:]}{m:02d}.parquet" for m in range(1, 13)}
    faltando = [m for m, arq in esperados.items() if not arq.exists()]
    if faltando:
        print(f"  {uf}/{ano}: faltam os meses {faltando} em data/raw; baixando…")
        nomes_faltando = {esperados[m].name.upper() for m in faltando}
        try:
            catalogo = list_files(dataset="sih", state=uf, year=ano)
            urls = [f"ftp://ftp.datasus.gov.br/{n}" for n in catalogo["name"]
                    if os.path.basename(str(n).replace("\\", "/")).upper() in nomes_faltando]
            copiados = []
            for origem in sih(state=uf, year=ano, month=faltando):
                origem = str(origem).replace("\\", "/")
                if os.path.basename(origem).upper() in nomes_faltando:
                    destino = RAIZ_BRUTOS / os.path.basename(origem)
                    shutil.copy(origem, destino)
                    copiados.append(destino)
            if copiados:
                registrar(
                    fonte="SIH", url="; ".join(urls),
                    metodo="sih (complemento de meses ausentes em data/raw)",
                    parametros={"state": uf, "year": ano, "month": faltando},
                    n_linhas=sum(pq.ParquetFile(a).metadata.num_rows for a in copiados),
                    arquivo_bruto=copiados,
                )
        except Exception as e:
            print(f"  ⚠ não consegui baixar {uf}/{ano} meses {faltando}: {e}")
    ainda_faltam = [m for m, arq in esperados.items() if not arq.exists()]
    return [arq for arq in esperados.values() if arq.exists()], ainda_faltam

In [ ]:
# SIH — internações por município de residência e mês de internação
partes, rd_sem_arquivo = [], {}
for uf in UFS:
    for ano in ANOS:
        arquivos, faltam = garantir_rd(uf, ano)
        if faltam:
            rd_sem_arquivo[f"{uf}/{ano}"] = faltam
        for arq in arquivos:
            partes.append(pd.read_parquet(arq, columns=["MUNIC_RES", "DT_INTER", "VAL_TOT", "MORTE"])
                          .assign(arquivo=arq.name))

rd = pd.concat(partes, ignore_index=True)
rd["codigo_ibge_6"] = codigo6(rd["MUNIC_RES"])
rd["ano_mes"] = para_mes(rd["DT_INTER"])
rd["VAL_TOT"] = pd.to_numeric(rd["VAL_TOT"], errors="coerce")
rd["MORTE"] = pd.to_numeric(rd["MORTE"], errors="coerce").fillna(0)
print(f"{len(rd):,} AIHs lidas de {rd['arquivo'].nunique()} arquivos RD "
      f"(esperado {len(UFS) * len(ANOS) * 12}); DT_INTER inválida: {rd['ano_mes'].isna().sum():,}")

dentro = rd["codigo_ibge_6"].isin(municipios["codigo_ibge_6"]) & rd["ano_mes"].isin(MESES)
print(f"Fora do recorte (residentes de outras UFs ou internação fora de 2022–2024): {(~dentro).sum():,}")

sih_mensal = (
    rd[dentro].groupby(["codigo_ibge_6", "ano_mes"])
    .agg(internacoes=("MUNIC_RES", "size"),
         custo_internacoes=("VAL_TOT", "sum"),
         obitos_hospitalares=("MORTE", "sum"))
    .reset_index()
)
if rd_sem_arquivo:
    print("⚠ Meses de processamento sem arquivo RD (os totais perto desses meses ficam baixos):", rd_sem_arquivo)

In [ ]:
# SIM — óbitos por município de residência e mês do óbito
partes = []
for uf in UFS:
    for ano in ANOS:
        arq = RAIZ_BRUTOS / f"DO{uf}{ano}.parquet"
        if not arq.exists():
            raise FileNotFoundError(f"{arq.name} não está em data/raw. Rode a coleta do SIM ou o git lfs pull.")
        partes.append(pd.read_parquet(arq, columns=["CODMUNRES", "DTOBITO"]))

do = pd.concat(partes, ignore_index=True)
do["codigo_ibge_6"] = codigo6(do["CODMUNRES"])
do["ano_mes"] = para_mes(do["DTOBITO"])
print(f"{len(do):,} declarações de óbito; DTOBITO inválida: {do['ano_mes'].isna().sum():,}")

dentro = do["codigo_ibge_6"].isin(municipios["codigo_ibge_6"]) & do["ano_mes"].isin(MESES)
sim_mensal = (
    do[dentro].groupby(["codigo_ibge_6", "ano_mes"])
    .size().rename("obitos_sim").reset_index()
)

In [ ]:
# SINAN — notificações por município de residência e mês dos primeiros sintomas.
# Os arquivos são nacionais e grandes (dengue 2024 tem milhões de linhas), então a agregação
# é feita direto no DuckDB, sem carregar tudo no pandas.
DOENCAS_SINAN = {"LEPT": "leptospirose", "DENG": "dengue", "ZIKAB": "zika", "CHIKB": "chikungunya"}

def data_sql(coluna):
    v = f"substr(CAST({coluna} AS VARCHAR), 1, 10)"
    return f"COALESCE(TRY_CAST({v} AS DATE), CAST(TRY_STRPTIME({v}, '%Y%m%d') AS DATE))"

sinan_partes, sinan_sem_arquivo = [], []
for sigla, nome in DOENCAS_SINAN.items():
    for ano in ANOS:
        arq = RAIZ_BRUTOS / f"sinan_{sigla.lower()}_{ano}.parquet"
        if not arq.exists():
            sinan_sem_arquivo.append((nome, ano))
            continue
        caminho_sql = str(arq).replace("\\", "/")
        parte = duckdb.sql(f"""
            SELECT TRY_CAST(substr(CAST(ID_MN_RESI AS VARCHAR), 1, 6) AS BIGINT) AS codigo_ibge_6,
                   strftime(COALESCE({data_sql('DT_SIN_PRI')}, {data_sql('DT_NOTIFIC')}), '%Y-%m') AS ano_mes,
                   count(*) AS n
            FROM read_parquet('{caminho_sql}')
            GROUP BY 1, 2
        """).df()
        sinan_partes.append(parte.assign(coluna=f"notificacoes_{nome}"))

sinan_long = pd.concat(sinan_partes, ignore_index=True)
sinan_long = sinan_long[sinan_long["codigo_ibge_6"].isin(municipios["codigo_ibge_6"])
                        & sinan_long["ano_mes"].isin(MESES)]
sinan_mensal = (
    sinan_long.pivot_table(index=["codigo_ibge_6", "ano_mes"], columns="coluna",
                           values="n", aggfunc="sum")
    .reset_index()
)
sinan_mensal.columns.name = None
if sinan_sem_arquivo:
    print("⚠ SINAN sem arquivo em data/raw (ficam vazios na tabela):", sinan_sem_arquivo)

In [ ]:
# Painel completo: todo município × todo mês, e as contagens entram por merge à esquerda
painel = municipios.merge(pd.DataFrame({"ano_mes": MESES}), how="cross")
for tabela in (sih_mensal, sim_mensal, sinan_mensal):
    painel = painel.merge(tabela, on=["codigo_ibge_6", "ano_mes"], how="left", validate="1:1")
painel.insert(painel.columns.get_loc("ano_mes") + 1, "ano", painel["ano_mes"].str[:4].astype(int))

COLS_SINAN = [f"notificacoes_{n}" for n in DOENCAS_SINAN.values()]
for col in COLS_SINAN:
    if col not in painel.columns:
        painel[col] = pd.NA
CONTAGENS = ["internacoes", "obitos_hospitalares", "obitos_sim"] + COLS_SINAN

# ausência de registro = 0 ...
painel[CONTAGENS + ["custo_internacoes"]] = painel[CONTAGENS + ["custo_internacoes"]].fillna(0)
# ... exceto doença/ano do SINAN que não foi coletado: aí fica vazio, não zero
for nome, ano in sinan_sem_arquivo:
    painel.loc[painel["ano"] == ano, f"notificacoes_{nome}"] = pd.NA
painel[CONTAGENS] = painel[CONTAGENS].astype("Int64")
painel = painel[["codigo_ibge", "codigo_ibge_6", "nome_municipio", "uf", "ano_mes", "ano",
                 "internacoes", "custo_internacoes", "obitos_hospitalares", "obitos_sim"] + COLS_SINAN]
painel = painel.sort_values(["uf", "codigo_ibge", "ano_mes"]).reset_index(drop=True)
painel.head()

## Conferências

In [ ]:
# 1) uma linha por município e mês, nada perdido no merge
assert len(painel) == len(municipios) * len(MESES)
assert not painel.duplicated(["codigo_ibge_6", "ano_mes"]).any()
assert painel["internacoes"].sum() == sih_mensal["internacoes"].sum()
assert painel["obitos_sim"].sum() == sim_mensal["obitos_sim"].sum()
print(f"✓ {len(painel):,} linhas = {len(municipios)} municípios × {len(MESES)} meses")

# 2) totais por UF e ano (para comparar com o TabNet do DATASUS)
print()
print(painel.groupby(["uf", "ano"])[CONTAGENS].sum().to_string())

# 3) série mensal por UF: buracos ou quedas bruscas indicam arquivo faltando
print()
print(painel.groupby(["ano_mes", "uf"])[["internacoes", "obitos_sim"]].sum().unstack("uf").to_string())

# 4) compatibilidade com o Atlas: todo município do Atlas tem que existir aqui
arq_atlas = TRATADOS / "atlas_eventos_am_rs_2022_2024.csv"
if arq_atlas.exists():
    atlas = pd.read_csv(arq_atlas)
    fora = set(atlas["codigo_ibge_6"]) - set(painel["codigo_ibge_6"])
    print("\nMunicípios do Atlas que não estão na tabela mensal:", sorted(fora))   # esperado: []
else:
    print("\n(atlas_eventos_am_rs_2022_2024.csv não está nesta branch; conferência com o Atlas pulada)")

## Salvar a tabela e conferir que a base original não mudou

In [ ]:
painel.to_csv(SAIDA_MENSAL.with_suffix(".csv"), index=False)
painel.to_parquet(SAIDA_MENSAL.with_suffix(".parquet"), index=False)
print("Salvo:", SAIDA_MENSAL.with_suffix(".csv").relative_to(RAIZ), "e .parquet —", len(painel), "linhas")

assert hash_arquivo(ARQ_BASE_ORIGINAL) == HASH_BASE_ORIGINAL, \
    "base_consolidada_municipios.csv mudou durante a execução!"
print("✓ base_consolidada_municipios.csv continua idêntica")

## Limitações

* **Contagem de AIHs, não de pessoas** (igual à base original): internações longas geram mais de uma AIH, e a mesma pessoa pode ser internada várias vezes.
* **Só hospitais do AM e do RS.** Os arquivos do SIH/SIM são por UF de ocorrência; um residente atendido ou falecido em outro estado não aparece.
* **Bordas do período.** As AIHs de internações do fim de 2024 processadas em 2025, e as notificações do SINAN de 2024 registradas em 2025, não foram baixadas, então os últimos meses de 2024 ficam um pouco subestimados. O mesmo acontece no começo de 2022 com o que foi processado em 2021.
* **SINAN = notificações**, inclusive casos depois descartados (a coluna `CLASSI_FIN` não foi coletada).
* **Não comparar direto com `total_internacoes` da base original.** A base original soma períodos diferentes por UF e conta as internações duas vezes (lê os `RD*` e os `sih_*` juntos). Esta tabela não tem esses problemas, mas por isso os números não batem com a base.
* Se aparecer o aviso de meses sem arquivo RD, esses meses de processamento faltam e os totais em volta deles ficam baixos.

In [96]:
%%script echo "Ignorando comandos de Git"
TOKEN = userdata.get('GITHUB_TOKEN')

!git remote set-url origin https://{TOKEN}@github.com/victorbaraunaAcad/crise-saude-ufam.git
!git checkout Gabriel-branch
!git add .
!git commit -m "Extensão dataSUS"

!git push -u origin Gabriel-branch

Ignorando comandos de Git
